# Divvy Bike-Share Case Study
How do annual members and casual riders use Divvy differently?

Full pipeline: load 12 monthly files, clean, verify, analyze, visualize.
Data: Sep 2025 - Aug 2026, ~6.1M rides after cleaning.

## 1. Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
plt.style.use("seaborn-v0_8-whitegrid")
os.makedirs("processed", exist_ok=True)
os.makedirs("analysis", exist_ok=True)
os.makedirs("figures", exist_ok=True)

## 2. Cleaning functions

In [ ]:
# Standardize column names across monthly files
def standardize(df):
    rename = {
        "rideable_type": "rideable_type",
        "started_at": "started_at",
        "ended_at": "ended_at",
        "member_casual": "user_type",
        "member": "user_type",
        "usertype": "user_type",
    }
    df = df.rename(columns=rename)
    return df

# Clean a single month
def clean(df, month_label):
    # drop rows with missing key fields
    df = df.dropna(subset=["ride_id", "started_at", "ended_at", "user_type"])
    # remove invalid or extreme durations
    df = df[(df["ride_length"] > 0) & (df["ride_length"] < 1440)]
    # remove duplicates within the month
    df = df.drop_duplicates(subset="ride_id")
    return df

# NOTE: replace the bodies above with your own functions if they differ.

## 3. Process all 12 months

In [ ]:
months = ["202509","202510","202511","202512",
          "202601","202602","202603",
          "202604","202605","202606","202607","202608"]

files_to_process = [f"raw/{m}-divvy-tripdata.csv" for m in months
                    if os.path.exists(f"raw/{m}-divvy-tripdata.csv")]

print(f"Files to process: {len(files_to_process)} of 12")

frames = []
for path in files_to_process:
    month_label = os.path.basename(path).split("-")[0]
    df = pd.read_csv(path)
    df = standardize(df)
    df["started_at"] = pd.to_datetime(df["started_at"])
    df["ended_at"] = pd.to_datetime(df["ended_at"])
    df["ride_length"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
    df["day_of_week"] = (df["started_at"].dt.dayofweek + 1) % 7 + 1
    df["day_name"] = df["started_at"].dt.day_name()
    df["month"] = df["started_at"].dt.month
    before = len(df)
    df = clean(df, month_label)
    print(f"{month_label}: {before:,} rows before -> {len(df):,} rows after")
    df["source_file"] = month_label
    frames.append(df)

all_trips = pd.concat(frames, ignore_index=True)

## 4. Cross-month deduplication and verification

In [ ]:
before = len(all_trips)
all_trips = all_trips.drop_duplicates(subset="ride_id")
print(f"Removed {before - len(all_trips)} cross-month duplicate ride_ids")

print("Rides with length <= 0:", (all_trips["ride_length"] <= 0).sum())
print("Rides longer than 24h:", (all_trips["ride_length"] >= 1440).sum())
print("Duplicate ride_ids:", all_trips["ride_id"].duplicated().sum())
print("Missing user_type:", all_trips["user_type"].isnull().sum())
print("Final shape:", all_trips.shape)

keep = ["ride_id", "started_at", "ended_at", "ride_length",
        "day_of_week", "day_name", "month", "user_type",
        "rideable_type", "start_station_name", "end_station_name",
        "source_file"]
all_trips[keep].to_csv("processed/divvy_2025_2026_cleaned.csv", index=False)
print("Saved: processed/divvy_2025_2026_cleaned.csv")

## 5. Analysis

In [ ]:
df = pd.read_csv("processed/divvy_2025_2026_cleaned.csv",
                 usecols=["ride_length", "day_of_week", "day_name",
                          "month", "user_type", "rideable_type",
                          "source_file"])

print("===== Ride length by user type =====")
print(df.groupby("user_type")["ride_length"].agg(
    count="count", mean="mean", median="median").round(1))

print("\n===== Rides by day of week =====")
pivot3 = df.pivot_table(values="ride_length", index="day_of_week",
                        columns="user_type", aggfunc="count")
share = (pivot3.div(pivot3.sum(axis=0), axis=1) * 100).round(1)
print(share)

print("\n===== Monthly rides =====")
rides_by_month = df.pivot_table(values="ride_length", index="month",
                                columns="user_type", aggfunc="count")
print(rides_by_month)

print("\n===== Monthly average length =====")
print(df.pivot_table(values="ride_length", index="month",
                     columns="user_type", aggfunc="mean").round(1))

print("\n===== Bike type =====")
bike_counts = df.pivot_table(values="ride_length", index="rideable_type",
                             columns="user_type", aggfunc="count")
print((bike_counts / bike_counts.sum(axis=0) * 100).round(1))

## 6. Export summary tables

In [ ]:
df.groupby("user_type")["ride_length"].agg(
    count="count", mean="mean", median="median").round(1) \
    .to_csv("analysis/pivot1_ride_length_by_user_type.csv")
df.pivot_table(values="ride_length", index="day_of_week",
               columns="user_type", aggfunc="mean").round(1) \
    .to_csv("analysis/pivot2_avg_length_by_day.csv")
pivot3.to_csv("analysis/pivot3_rides_by_day.csv")
rides_by_month.to_csv("analysis/monthly_ride_counts.csv")
df.pivot_table(values="ride_length", index="month",
               columns="user_type", aggfunc="mean").round(1) \
    .to_csv("analysis/monthly_avg_length.csv")
bike_counts.to_csv("analysis/bike_type_counts.csv")
print("Exported 6 files to analysis/")

## 7. Visualizations

In [ ]:
# Chart 1: rides by day of week
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
day_counts = df.groupby(["day_name","user_type"]).size().unstack().reindex(order)
ax = day_counts.plot(kind="bar", figsize=(10,5), color=["#e07a5f","#3d5a80"])
ax.set_title("Rides by Day of Week - Members vs Casual")
ax.set_ylabel("Number of rides"); ax.set_xlabel("")
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig("figures/rides_by_day.png", dpi=150); plt.show()

# Chart 2: average ride length
avg_len = df.groupby("user_type")["ride_length"].agg(["mean","median"]).T
ax = avg_len.plot(kind="bar", figsize=(7,5), color=["#e07a5f","#3d5a80"])
ax.set_title("Average vs Median Ride Length (minutes)")
ax.set_ylabel("Minutes"); ax.set_xlabel("")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("figures/ride_length.png", dpi=150); plt.show()

# Chart 3: monthly seasonality
monthly = df.pivot_table(values="ride_length", index="month",
                         columns="user_type", aggfunc="count")
ax = monthly.plot(kind="line", marker="o", figsize=(10,5),
                  color=["#e07a5f","#3d5a80"])
ax.set_title("Rides by Calendar Month - Seasonality")
ax.set_ylabel("Number of rides")
ax.set_xlabel("Calendar month (Jan-Aug = 2026, Sep-Dec = 2025)")
ax.set_xticks(range(1,13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"])
plt.tight_layout()
plt.savefig("figures/monthly_rides.png", dpi=150); plt.show()

# Chart 4: bike type
ax = bike_counts.plot(kind="bar", figsize=(7,5), color=["#e07a5f","#3d5a80"])
ax.set_title("Rides by Bike Type")
ax.set_ylabel("Number of rides"); ax.set_xlabel("")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("figures/bike_type.png", dpi=150); plt.show()

print("All four charts saved to figures/")

## 8. Conclusions

- Members: 3.95M rides, 12-min average, midweek peaks, stable across seasons - commuters.
- Casual: 2.16M rides, 18-min average, weekend peaks, 14-fold seasonal swing - leisure riders.
- Both prefer electric bikes (casuals slightly more).

Recommendations and full report: see REPORT.md.